# SPM 405: Machine Learning — Lecture 7 notebook
## Neural networks and deep learning

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will, using **PyTorch**:
1. Train fully connected networks on the six-sensor machines, watch them **overfit**, and fix it with **dropout**.
2. Classify thermal images of circuit boards with an ordinary network and a **convolutional network (CNN)**.
3. Forecast a building's hourly demand with a linear **AR(24)** model and a **recurrent network (GRU)**.

Reading: ISLP 10.1–10.7. Runs on a CPU in about a minute. Numbers can differ slightly from the slides with other PyTorch versions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score

torch.set_num_threads(2)
rng = np.random.default_rng(405)   # fixed seed: everyone gets the same data as the slides
count_params = lambda net: sum(p.numel() for p in net.parameters())

## Part 1: Six-sensor machines (the Lecture 5 rules, 2000 training machines)

In [ ]:
def plant(m):
    vib = rng.gamma(4, 0.8, m); temp = rng.normal(55, 10, m); cur = rng.normal(10, 2, m)
    noise_db = rng.normal(70, 5, m); hours = rng.uniform(0, 20000, m); hum = rng.uniform(20, 90, m)
    risk = (vib > 4.5) | ((temp > 65) & (cur > 11)) | ((hours > 15000) & (noise_db > 75))
    flip = rng.uniform(size=m) < 0.05
    return np.c_[vib, temp, cur, noise_db, hours, hum], (risk ^ flip).astype(int)

X_train, y_train = plant(2000); X_test, y_test = plant(5000)
sc = StandardScaler().fit(X_train); Z_train, Z_test = sc.transform(X_train), sc.transform(X_test)

p = LogisticRegression(max_iter=1000).fit(Z_train, y_train).predict_proba(Z_test)[:, 1]
print(f"logistic regression: test error {np.mean((p >= .5) != y_test):.3f}")
p = RandomForestClassifier(300, random_state=0).fit(X_train, y_train).predict_proba(X_test)[:, 1]
print(f"random forest:       test error {np.mean((p >= .5) != y_test):.3f}")

In [ ]:
def train_mlp(hidden, epochs=200, lr=1e-2, dropout=0.0, seed=0, log=False):
    torch.manual_seed(seed)
    layers, d = [], 6
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]; d = h
    layers += [nn.Linear(d, 1)]                          # one output unit: the log-odds of a fault
    net = nn.Sequential(*layers)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()                     # cross-entropy (Lecture 3)
    X = torch.tensor(Z_train, dtype=torch.float32); Y = torch.tensor(y_train, dtype=torch.float32)[:, None]
    XT = torch.tensor(Z_test, dtype=torch.float32)
    g = torch.Generator().manual_seed(seed); history = []
    for epoch in range(epochs):
        net.train(); perm = torch.randperm(len(X), generator=g)
        for i in range(0, len(X), 64):                   # minibatches of 64
            b = perm[i:i + 64]
            opt.zero_grad(); loss = loss_fn(net(X[b]), Y[b]); loss.backward(); opt.step()
        if log and (epoch + 1) % 10 == 0:
            net.eval()
            with torch.no_grad():
                history.append((epoch + 1, loss_fn(net(X), Y).item(),
                                np.mean((torch.sigmoid(net(XT)).numpy()[:, 0] >= .5) != y_test)))
    net.eval()
    with torch.no_grad():
        p_test = torch.sigmoid(net(XT)).numpy()[:, 0]; p_train = torch.sigmoid(net(X)).numpy()[:, 0]
    return net, p_test, p_train, history

for hidden, dr in [((), 0.0), ((16,), 0.0), ((64, 64), 0.0), ((64, 64), 0.3)]:
    net, p, ptr, _ = train_mlp(hidden, dropout=dr)
    print(f"hidden {str(hidden):9s} dropout {dr}: {count_params(net):5d} parameters   "
          f"train error {np.mean((ptr >= .5) != y_train):.3f}   test error {np.mean((p >= .5) != y_test):.3f}")

In [ ]:
net, p, ptr, hist = train_mlp((64, 64), log=True)
hist = np.array(hist)
plt.plot(hist[:, 0], hist[:, 1], label="training loss")
plt.plot(hist[:, 0], hist[:, 2], label="test error")
plt.xlabel("epoch"); plt.legend(); plt.show()

**Questions**
1. With no hidden layer, the network has 7 parameters. Which model from Lecture 3 is it? Compare its test error with logistic regression.
2. Check the count of 4673 parameters for the 64 + 64 network by hand.
3. From the plot, at which epoch would early stopping stop? What test error would you get?
4. Why does the random forest still win on this data?

## Part 2: Thermal images of circuit boards (24 × 24 pixels)

In [ ]:
S = 24
def boards(m):
    y = (rng.uniform(size=m) < 0.5).astype(int)                  # 1 = faulty (hot spot)
    yy, xx = np.mgrid[0:S, 0:S]
    imgs = np.empty((m, S, S), dtype=np.float32)
    for i in range(m):
        base = 30 + 3 * rng.uniform() * (xx / S) + 3 * rng.uniform() * (yy / S)   # warm board (deg C)
        for _ in range(2):                                        # two larger warm components on every board
            cx, cy = rng.uniform(3, S - 3, 2); base = base + 1.5 * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * 3.0**2))
        if y[i] == 1:                                             # small hot spot, anywhere
            cx, cy = rng.uniform(3, S - 3, 2); base = base + 2.5 * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * 1.0**2))
        imgs[i] = base + rng.normal(0, 0.7, (S, S))
    return imgs, y

I_train, yi_train = boards(3000); I_test, yi_test = boards(2000)
mu, sd = I_train.mean(), I_train.std()
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
for a, k, t in zip(ax, [0, 1], ["normal", "faulty"]):
    a.imshow(I_train[np.where(yi_train == k)[0][0]], cmap="hot"); a.set_title(t); a.axis("off")
plt.show()

In [ ]:
def fit_images(net, epochs=15, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3); loss_fn = nn.BCEWithLogitsLoss()
    X = torch.tensor((I_train - mu) / sd)[:, None]; Y = torch.tensor(yi_train, dtype=torch.float32)[:, None]
    XT = torch.tensor((I_test - mu) / sd)[:, None]
    g = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train(); perm = torch.randperm(len(X), generator=g)
        for i in range(0, len(X), 64):
            b = perm[i:i + 64]; opt.zero_grad(); loss_fn(net(X[b]), Y[b]).backward(); opt.step()
    net.eval()
    with torch.no_grad():
        return torch.sigmoid(net(XT)).numpy()[:, 0], torch.sigmoid(net(X)).numpy()[:, 0]

torch.manual_seed(0)
fc = nn.Sequential(nn.Flatten(), nn.Linear(S * S, 128), nn.ReLU(), nn.Linear(128, 1))
p, ptr = fit_images(fc)
print(f"ordinary network: {count_params(fc)} parameters  train {np.mean((ptr >= .5) != yi_train):.3f}  "
      f"test {np.mean((p >= .5) != yi_test):.3f}  AUC {roc_auc_score(yi_test, p):.2f}")

torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 8 filters of 3x3 -> 8 x 12 x 12
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # -> 16 x 6 x 6
    nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool2d(1),   # strongest response anywhere -> 16
    nn.Flatten(), nn.Linear(16, 1))
p, ptr = fit_images(cnn)
print(f"CNN:              {count_params(cnn)} parameters  train {np.mean((ptr >= .5) != yi_train):.3f}  "
      f"test {np.mean((p >= .5) != yi_test):.3f}  AUC {roc_auc_score(yi_test, p):.2f}")
print("hottest-pixel rule AUC:", round(roc_auc_score(yi_test, I_test.reshape(len(I_test), -1).max(1)), 2))

**Questions**
5. Why does the ordinary network reach training error 0.02 but test error close to 0.5?
6. Where do the CNN's 3585 parameters come from? (A 3 × 3 convolution from $a$ to $b$ channels has $9ab + b$ parameters.)
7. Replace `nn.AdaptiveMaxPool2d(1)` by `nn.AdaptiveAvgPool2d(1)`. What happens, and why? (Think about a small spot in a large image.)

## Part 3: Forecasting hourly demand

In [ ]:
hours = 24 * 200
hh = np.arange(hours) % 24; day = np.arange(hours) // 24
temp = 24 + 4 * np.sin(2 * np.pi * day / 60) + np.repeat(rng.normal(0, 2, 200), 24) + 3 * np.sin(2 * np.pi * (hh - 9) / 24)
series = 20 + 25 * np.exp(-0.5 * ((hh - 9) / 2)**2) + 35 * np.exp(-0.5 * ((hh - 19.5) / 1.8)**2) + 0.8 * (temp - 22)
e = 0.0
for t in range(hours):                           # disturbances that last a few hours
    e = 0.7 * e + rng.normal(0, 3); series[t] += e

L = 24                                           # use the last 24 hours
X = np.stack([series[t - L:t] for t in range(L, hours)]); y = series[L:]
n_train = 24 * 150 - L                           # train on the first 150 days, test on the last 50
Xa, ya, Xb, yb = X[:n_train], y[:n_train], X[n_train:], y[n_train:]
mse = lambda a, b: np.mean((a - b)**2)
print("same hour yesterday: ", round(mse(Xb[:, 0], yb), 1))
ar = LinearRegression().fit(Xa, ya)
print("linear AR(24):       ", round(mse(ar.predict(Xb), yb), 1))

In [ ]:
class GRUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=16, batch_first=True)
        self.out = nn.Linear(16, 1)
    def forward(self, x):
        h, _ = self.gru(x)                       # h: the hidden state after every hour
        return self.out(h[:, -1])                # forecast from the last hidden state

m_, s_ = ya.mean(), ya.std()
XA = torch.tensor((Xa - m_) / s_, dtype=torch.float32)[:, :, None]; YA = torch.tensor((ya - m_) / s_, dtype=torch.float32)[:, None]
XB = torch.tensor((Xb - m_) / s_, dtype=torch.float32)[:, :, None]
torch.manual_seed(0); gru = GRUNet()
opt = torch.optim.Adam(gru.parameters(), lr=3e-3); loss_fn = nn.MSELoss()
g = torch.Generator().manual_seed(0)
for epoch in range(40):
    perm = torch.randperm(len(XA), generator=g)
    for i in range(0, len(XA), 64):
        b = perm[i:i + 64]; opt.zero_grad(); loss_fn(gru(XA[b]), YA[b]).backward(); opt.step()
with torch.no_grad():
    pg = gru(XB).numpy()[:, 0] * s_ + m_
print("GRU:                 ", round(mse(pg, yb), 1), f"({count_params(gru)} parameters)")

plt.figure(figsize=(10, 3))
plt.plot(yb[:96], "k", label="actual"); plt.plot(ar.predict(Xb)[:96], "--", label="AR(24)"); plt.plot(pg[:96], label="GRU")
plt.xlabel("hour of test period"); plt.ylabel("demand (kW)"); plt.legend(); plt.show()

**Questions**
8. The disturbance has variance 9 kW² each hour. Why can no forecast beat a test MSE of about 9?
9. Why must a forecasting model be trained on the past and tested on the future, not on a random split?
10. **Your group project:** is your data a table, images, signals or sequences? Which kind of network (if any) would suit it?

## Your turn: group tasks

Work on these in your group. Write the code in the empty cells and answer each question in a text cell below it,
in your own words. Submit **one notebook per group** on Google Classroom, with every cell run.
Every member must be able to explain every line.

The tasks reuse `Z_train`, `y_train`, `Z_test`, `y_test`, `I_train`, `yi_train`, `I_test`, `yi_test`, `mu`, `sd`, `S` and
`count_params` from above, so run the whole notebook first. Everything runs on a CPU in a few minutes.

### Task 1: Early stopping, done properly

In Question 3 you chose the stopping epoch by looking at the **test** error. That is cheating: the test data must not be used to
make choices. Use a validation set instead.

(a) Split the 2000 training machines into 1500 for fitting and 500 for validation.

(b) Write a training loop for the `(64, 64)` network (no dropout; Adam, learning rate 0.01, minibatches of 64) that runs for 200 epochs
and, after every epoch, computes the **validation** cross-entropy. Keep a copy of the weights with the lowest validation loss
(`copy.deepcopy(net.state_dict())`).

(c) Plot the training and validation loss against the epoch. At which epoch is the validation loss lowest? Load the best weights
(`net.load_state_dict(...)`) and give the test error. Compare with the 200-epoch network of Part 1.

In [ ]:
# Task 1: your code here

### Task 2: Representation matters — the hour of the day

A network predicts a building's electricity demand from the hour of the day and the outdoor temperature (the Lecture 5 building).
Hour 23 and hour 0 are one hour apart, but as numbers they are 23 apart. The cell below makes the data; do not change it.

(a) Train a network with one hidden layer of 32 units (`nn.MSELoss`, Adam, learning rate 0.01, 300 epochs, minibatches of 64) on
the inputs (hour, temperature), standardised. Give its test MSE.

(b) Now represent the hour by **two** inputs, $\sin(2\pi h/24)$ and $\cos(2\pi h/24)$, plus the temperature. Train the same network and
give its test MSE.

(c) Plot both networks' predictions against the hour at 24 °C, with the truth. Why does the second representation help?
Which of the three parts of ML from Lecture 1 is this?

In [ ]:
rng7 = np.random.default_rng(4060)                # task data: do not change
def building(m):
    h = rng7.uniform(0, 24, m); t = rng7.normal(24, 4, m)
    d = 20 + 25 * np.exp(-0.5 * ((h - 9) / 2)**2) + 35 * np.exp(-0.5 * ((h - 19.5) / 1.8)**2) \
        + 30 * np.exp(-0.5 * ((h - 0.5) / 1.5)**2) + 30 * np.exp(-0.5 * ((h - 24.5) / 1.5)**2) + 0.8 * (t - 22)   # night pumping near midnight
    return h, t, d + rng7.normal(0, 4, m)
hb, tb, db = building(600)
hbt, tbt, dbt = building(3000)

In [ ]:
# Task 2: your code here

### Task 3: How much data does each network need?

(a) Write a function `image_errors(n)` that trains a fresh ordinary network and a fresh CNN (the same layers as in Part 2) on only the
first `n` training boards and returns both test errors. (Copy `fit_images` and change it to use the first `n` boards.)
So that every run gets the same number of weight updates, train for `15 * 3000 // n` epochs.

(b) Run it for `n` = 300, 1000 and 3000 and show the results in a table.

(c) Which network needs less data, and why? (Think about how many parameters each has, and what a convolution assumes about images.)

In [ ]:
# Task 3: your code here